# ML Training Workflow

This notebook demonstrates the complete ML training workflow using the Makalu training library:

1. **Load Data** - Load features and labels from parquet files
2. **Explore Data** - Understand the data distribution and quality
3. **Preprocess Data** - Handle missing values and prepare for training
4. **Train Model** - Train a LightGBM model
5. **Evaluate Model** - Assess model performance
6. **Analyze Results** - Deep dive into predictions and feature importance

## Setup

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from training import Dataset, LGBMTrainer, Evaluator

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
%matplotlib inline

print("Setup complete!")

## 1. Load Data

Load features and labels from parquet files generated by the ML pipeline.

In [ ]:
# Configure paths - update these to your actual file paths
FEATURES_PATH = "/path/to/your/market_id_features.parquet"
LABELS_PATH = "/path/to/your/market_id_labels.parquet"

# Target column for prediction
TARGET_COLUMN = "direction_label"  # 1 = profit, 0 = loss

In [ ]:
# Load dataset
dataset = Dataset.load(
    features_path=FEATURES_PATH,
    labels_path=LABELS_PATH,
    target_column=TARGET_COLUMN
)

print(f"Dataset loaded successfully!")

## 2. Explore Data

Understand the data distribution, quality, and characteristics.

In [ ]:
# Get dataset summary
summary = dataset.summary()

print(f"Total rows: {summary['total_rows']}")
print(f"Number of features: {summary['num_features']}")
print(f"Target column: {summary['target_column']}")
print(f"\nTarget distribution:")
for k, v in summary['target_distribution'].items():
    print(f"  {k}: {v}")

In [ ]:
# Get aligned data for exploration
X, y = dataset.get_aligned_data()

print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"\nFeature columns:")
print(X.columns.tolist())

In [ ]:
# Check for missing values
na_counts = X.isna().sum()
na_pct = (X.isna().sum() / len(X) * 100).round(2)

na_summary = pd.DataFrame({
    'na_count': na_counts,
    'na_pct': na_pct
}).sort_values('na_pct', ascending=False)

print("Columns with missing values:")
print(na_summary[na_summary['na_count'] > 0].head(20))

In [ ]:
# Feature statistics
X.describe().T

In [ ]:
# Target distribution visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar plot of target distribution
y.value_counts().plot(kind='bar', ax=axes[0])
axes[0].set_title('Target Distribution')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')

# Time series of target
y.rolling(100).mean().plot(ax=axes[1])
axes[1].set_title('Target Rolling Mean (window=100)')
axes[1].set_xlabel('Time')
axes[1].set_ylabel('Mean')

plt.tight_layout()
plt.show()

In [ ]:
# Feature correlation with target (top 20)
correlations = X.corrwith(y).abs().sort_values(ascending=False)

print("Top 20 features correlated with target:")
print(correlations.head(20))

In [ ]:
# Correlation heatmap of top features
top_features = correlations.head(15).index.tolist()

plt.figure(figsize=(12, 10))
corr_matrix = X[top_features].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix (Top 15)')
plt.tight_layout()
plt.show()

## 3. Preprocess Data

Handle missing values and prepare data for training.

In [ ]:
# Drop columns with too many missing values (>50%)
dataset.drop_na_features(threshold=0.5)

# Fill remaining missing values
dataset.fill_na(method='ffill')

# Check result
X, y = dataset.get_aligned_data()
print(f"After preprocessing: {X.shape[0]} rows, {X.shape[1]} features")
print(f"Remaining NaN values: {X.isna().sum().sum()}")

In [ ]:
# Split into train/test sets
# For time-series, we don't shuffle - train on past, test on future
dataset.train_test_split(test_size=0.2, shuffle=False)

X_train, y_train = dataset.get_train_data()
X_test, y_test = dataset.get_test_data()

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")
print(f"\nTrain period: {X_train.index.min()} to {X_train.index.max()}")
print(f"Test period: {X_test.index.min()} to {X_test.index.max()}")

In [ ]:
# Check class balance in train/test
print("Class distribution:")
print(f"  Train: {y_train.value_counts().to_dict()}")
print(f"  Test: {y_test.value_counts().to_dict()}")

## 4. Train Model

Train a LightGBM classifier with early stopping.

In [ ]:
# Initialize trainer with custom parameters (optional)
trainer = LGBMTrainer(
    task="classification",
    params={
        "objective": "binary",
        "metric": "binary_logloss",
        "boosting_type": "gbdt",
        "num_leaves": 31,
        "learning_rate": 0.05,
        "feature_fraction": 0.9,
        "bagging_fraction": 0.8,
        "bagging_freq": 5,
        "verbose": -1,
        "n_jobs": -1,
        "random_state": 42,
    },
    num_boost_round=1000,
    early_stopping_rounds=50,
    verbose_eval=100,
)

print("Trainer initialized with parameters:")
print(trainer.get_params())

In [ ]:
# Train model with validation set for early stopping
trainer.fit(
    X_train, y_train,
    X_val=X_test, y_val=y_test
)

print(f"\nTraining complete!")
print(f"Best iteration: {trainer.best_iteration}")

In [ ]:
# Plot training history
if trainer.training_history:
    fig, ax = plt.subplots(figsize=(10, 5))
    
    for name, values in trainer.training_history.items():
        for metric, metric_values in values.items():
            ax.plot(metric_values, label=f"{name}_{metric}")
    
    ax.axvline(x=trainer.best_iteration, color='r', linestyle='--', label=f'Best iteration ({trainer.best_iteration})')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Loss')
    ax.set_title('Training History')
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Save model
MODEL_PATH = "../models/lgbm_model"
trainer.save(MODEL_PATH)
print(f"Model saved to {MODEL_PATH}")

## 5. Evaluate Model

Assess model performance on the test set.

In [ ]:
# Create evaluator
evaluator = Evaluator(trainer, threshold=0.5)

# Evaluate on test set
results = evaluator.evaluate(X_test, y_test, return_predictions=True)

# Print summary
print(results.summary())

In [ ]:
# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(
    results.confusion_matrix,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Predicted 0', 'Predicted 1'],
    yticklabels=['Actual 0', 'Actual 1']
)
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Plot ROC curve
if results.roc_curve:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # ROC curve
    axes[0].plot(results.roc_curve['fpr'], results.roc_curve['tpr'], 'b-', label=f"ROC (AUC = {results.metrics['roc_auc']:.4f})")
    axes[0].plot([0, 1], [0, 1], 'k--', label='Random')
    axes[0].set_xlabel('False Positive Rate')
    axes[0].set_ylabel('True Positive Rate')
    axes[0].set_title('ROC Curve')
    axes[0].legend()
    
    # Precision-Recall curve
    if results.pr_curve:
        axes[1].plot(results.pr_curve['recall'], results.pr_curve['precision'], 'b-')
        axes[1].set_xlabel('Recall')
        axes[1].set_ylabel('Precision')
        axes[1].set_title('Precision-Recall Curve')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Find best threshold
threshold_results = evaluator.find_best_threshold(X_test, y_test, metric='f1')

print(f"Best threshold: {threshold_results['best_threshold']:.2f}")
print(f"Metrics at best threshold:")
for k, v in threshold_results['best_metrics'].items():
    print(f"  {k}: {v:.4f}")

In [ ]:
# Plot threshold analysis
threshold_df = threshold_results['all_results']

plt.figure(figsize=(10, 6))
for col in ['accuracy', 'precision', 'recall', 'f1']:
    plt.plot(threshold_df['threshold'], threshold_df[col], label=col)

plt.axvline(x=threshold_results['best_threshold'], color='r', linestyle='--', label='Best F1 threshold')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Metrics vs Classification Threshold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Analyze Results

Deep dive into feature importance and prediction analysis.

In [ ]:
# Feature importance
importance_df = trainer.feature_importance(importance_type='gain')

print("Top 20 Important Features:")
print(importance_df.head(20))

In [ ]:
# Plot feature importance
top_n = 20
top_importance = importance_df.head(top_n)

plt.figure(figsize=(10, 8))
plt.barh(range(len(top_importance)), top_importance['importance'].values)
plt.yticks(range(len(top_importance)), top_importance['feature'].values)
plt.xlabel('Importance (Gain)')
plt.title(f'Top {top_n} Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Analyze predictions
analysis = evaluator.analyze_predictions(X_test, y_test)

print("Prediction Analysis Summary:")
print(f"\nCategory distribution:")
print(analysis['category'].value_counts())

print(f"\nAccuracy by confidence level:")
analysis['confidence_bin'] = pd.cut(analysis['confidence'], bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0])
print(analysis.groupby('confidence_bin')['correct'].agg(['mean', 'count']))

In [ ]:
# Plot probability distribution by actual class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of probabilities
for label in [0, 1]:
    mask = analysis['y_true'] == label
    axes[0].hist(analysis.loc[mask, 'probability'], bins=50, alpha=0.5, label=f'Class {label}')
axes[0].set_xlabel('Predicted Probability')
axes[0].set_ylabel('Count')
axes[0].set_title('Probability Distribution by Actual Class')
axes[0].legend()

# Calibration plot
from sklearn.calibration import calibration_curve
prob_true, prob_pred = calibration_curve(analysis['y_true'], analysis['probability'], n_bins=10)
axes[1].plot(prob_pred, prob_true, 's-', label='Model')
axes[1].plot([0, 1], [0, 1], 'k--', label='Perfectly calibrated')
axes[1].set_xlabel('Mean Predicted Probability')
axes[1].set_ylabel('Fraction of Positives')
axes[1].set_title('Calibration Curve')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Predictions over time
analysis_with_time = analysis.copy()
analysis_with_time.index = X_test.index

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Actual vs Predicted
axes[0].plot(analysis_with_time.index, analysis_with_time['y_true'].rolling(50).mean(), label='Actual (rolling)', alpha=0.7)
axes[0].plot(analysis_with_time.index, analysis_with_time['probability'].rolling(50).mean(), label='Predicted (rolling)', alpha=0.7)
axes[0].set_ylabel('Value')
axes[0].set_title('Actual vs Predicted (Rolling Mean)')
axes[0].legend()

# Rolling accuracy
rolling_acc = analysis_with_time['correct'].rolling(100).mean()
axes[1].plot(analysis_with_time.index, rolling_acc)
axes[1].axhline(y=0.5, color='r', linestyle='--', alpha=0.5)
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Rolling Accuracy (window=100)')

# Confidence over time
axes[2].plot(analysis_with_time.index, analysis_with_time['confidence'].rolling(100).mean())
axes[2].set_ylabel('Confidence')
axes[2].set_title('Rolling Confidence (window=100)')
axes[2].set_xlabel('Time')

plt.tight_layout()
plt.show()

## 7. Cross-Validation (Optional)

Perform time-series cross-validation for more robust evaluation.

In [ ]:
# Time-series cross-validation
X, y = dataset.get_aligned_data()

cv_results = evaluator.cross_validate(X, y, n_splits=5)

print("\nCross-Validation Summary:")
cv_summary = pd.DataFrame(cv_results)
print(cv_summary.describe().T[['mean', 'std']])

In [ ]:
# Plot CV results
fig, ax = plt.subplots(figsize=(10, 6))

cv_summary = pd.DataFrame(cv_results)
cv_summary.plot(kind='box', ax=ax)
ax.set_ylabel('Score')
ax.set_title('Cross-Validation Results Distribution')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 8. Load Saved Model (Verification)

Verify that the saved model can be loaded and produces the same results.

In [ ]:
# Load saved model
loaded_trainer = LGBMTrainer.load(MODEL_PATH)

print(f"Loaded model from {MODEL_PATH}")
print(f"Best iteration: {loaded_trainer.best_iteration}")
print(f"Number of features: {len(loaded_trainer.feature_names)}")

In [ ]:
# Verify predictions match
original_preds = trainer.predict(X_test)
loaded_preds = loaded_trainer.predict(X_test)

predictions_match = np.allclose(original_preds, loaded_preds)
print(f"Predictions match: {predictions_match}")

if predictions_match:
    print("✓ Model saved and loaded successfully!")
else:
    print("✗ Warning: Predictions differ after loading")

## Summary

This notebook demonstrated the complete ML workflow:

1. ✅ Loaded features and labels from parquet files
2. ✅ Explored data distribution and quality
3. ✅ Preprocessed data (handled missing values, split train/test)
4. ✅ Trained LightGBM model with early stopping
5. ✅ Evaluated model with comprehensive metrics
6. ✅ Analyzed feature importance and predictions
7. ✅ Performed cross-validation
8. ✅ Verified model save/load functionality

### Next Steps

- Tune hyperparameters using grid search or Optuna
- Try different feature sets
- Experiment with different target columns (pnl_return, etc.)
- Deploy model for inference